# CLIP Image Feature Extraction

This notebook implements Steps 7–9 of the Flickr8k preprocessing pipeline:

1. Load and verify dataset splits
2. Load Flickr8k images
3. Preprocess images for CLIP
4. Load and freeze CLIP image encoder
5. Extract image embeddings
6. Save precomputed CLIP features

In [6]:
# IMPORTS

import json
import torch

from pathlib import Path
from PIL import Image

from config import *

In [ ]:
# PROJECT SETUP

setup_directories()

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_ROOT    :", DATA_ROOT)
print("SPLIT_DIR    :", SPLIT_DIR)
print("SUBSET_DIR   :", SUBSET_DIR)
print("FEATURE_DIR  :", FEATURE_DIR)
print("METADATA_DIR :", METADATA_DIR)


print("\nDirectory existence:")

print(
    "SPLIT_DIR exists   :",
    SPLIT_DIR.exists()
)

print(
    "SUBSET_DIR exists  :",
    SUBSET_DIR.exists()
)

print(
    "FEATURE_DIR exists :",
    FEATURE_DIR.exists()
)

PROJECT_ROOT : d:\zfs-clip-image-captioning
DATA_ROOT    : d:\zfs-clip-image-captioning\notebook\data\flickr8k
SPLIT_DIR    : d:\zfs-clip-image-captioning\notebook\data\flickr8k\splits
SUBSET_DIR   : d:\zfs-clip-image-captioning\notebook\data\flickr8k\subsets
FEATURE_DIR  : d:\zfs-clip-image-captioning\notebook\data\flickr8k\features
METADATA_DIR : d:\zfs-clip-image-captioning\notebook\data\flickr8k\metadata

Directory existence:
SPLIT_DIR exists   : True
SUBSET_DIR exists  : True
FEATURE_DIR exists : True


In [ ]:
# INSPECT DATA SPLIT FILES

split_files = sorted(
    SPLIT_DIR.glob("*.json")
)

print("Split files:")

for file_path in split_files:

    print(
        "-",
        file_path.name
    )


required_files = {
    "train.json",
    "val.json",
    "test.json",
}

found_files = {
    file_path.name
    for file_path in split_files
}


assert required_files.issubset(
    found_files
), "Missing train / val / test split files"


print(
    "\nAll required split files found."
) 

Split files:
- test.json
- train.json
- val.json

All required split files found.


In [9]:
# JSON LOADER

def load_json(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)

In [10]:
# LOAD TRAIN / VALIDATION / TEST

train_data = load_json(
    SPLIT_DIR / "train.json"
)

val_data = load_json(
    SPLIT_DIR / "val.json"
)

test_data = load_json(
    SPLIT_DIR / "test.json"
)


print(
    f"Train      : {len(train_data):,} images"
)

print(
    f"Validation : {len(val_data):,} images"
)

print(
    f"Test       : {len(test_data):,} images"
)

Train      : 6,464 images
Validation : 808 images
Test       : 809 images


In [11]:
# INSPECT DATA STRUCTURE

sample_image_id = next(
    iter(train_data)
)

sample_captions = train_data[
    sample_image_id
]


print(
    "Image ID:",
    sample_image_id
)

print(
    "Number of captions:",
    len(sample_captions)
)


print("\nCaptions:")

for index, caption in enumerate(
    sample_captions,
    start=1
):

    print(
        f"{index}. {caption}"
    )

Image ID: 3472540184_b0420b921a.jpg
Number of captions: 5

Captions:
1. An older boy chases a laughing younger boy on the grass .
2. Two boys are running ; one 's smiling and being touched by the other .
3. Two children run and play in the grass .
4. Two young boys are running through a grassy area .
5. Two young boys run across a green yard .


In [12]:
# VERIFY DATA SPLITS

train_ids = set(
    train_data.keys()
)

val_ids = set(
    val_data.keys()
)

test_ids = set(
    test_data.keys()
)


assert train_ids.isdisjoint(
    val_ids
)

assert train_ids.isdisjoint(
    test_ids
)

assert val_ids.isdisjoint(
    test_ids
)


print(
    "Train / Validation / Test are disjoint."
)

Train / Validation / Test are disjoint.


In [13]:
# COLLECT ALL UNIQUE IMAGE IDS

all_image_ids = sorted(
    train_ids
    | val_ids
    | test_ids
)


print(
    f"Total unique images: "
    f"{len(all_image_ids):,}"
)

Total unique images: 8,081
